In [1]:

%%writefile tools/claim_merge.py
"""Merge equivalent claims across sub-questions BEFORE fact-checking (Day 24).

Why: the same fact often appears under several sub-questions, each time backed by a
single snippet. Checked separately, each copy has 1 source. Merged, the copies pool
their sources, so real corroboration becomes visible. No LLM call is used (Groq's
output-token budget is the bottleneck), only deterministic text rules.
"""
import re
from typing import Dict, List

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "at", "to", "for", "and", "or", "is", "are", "was",
    "were", "be", "been", "by", "with", "as", "that", "this", "it", "its", "from", "has",
    "have", "had", "which", "their", "also", "than", "then", "into", "over", "about",
}
NEGATIONS = {"not", "no", "never", "without", "cannot", "none", "neither", "nor"}
MIN_CONTENT_TOKENS = 4      # shorter claims are never merged (too little to compare safely)
JACCARD_THRESHOLD = 0.6
CONTAINMENT_THRESHOLD = 0.85


def _stem(w: str) -> str:
    return w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w


def _profile(claim: str) -> Dict:
    text = claim.lower().replace(",", "")
    words = re.findall(r"\d+(?:\.\d+)?|[a-z]+", text)
    numbers = frozenset(w for w in words if w[0].isdigit())
    negated = any(w in NEGATIONS for w in words)
    tokens = frozenset(_stem(w) for w in words
                       if not w[0].isdigit() and w not in STOPWORDS and w not in NEGATIONS)
    return {"numbers": numbers, "negated": negated, "tokens": tokens}


def claims_equivalent(a: str, b: str) -> bool:
    pa, pb = _profile(a), _profile(b)
    if pa["numbers"] != pb["numbers"] or pa["negated"] != pb["negated"]:
        return False   # "95%" vs "70%", or a claim and its negation, are NOT the same claim
    ta, tb = pa["tokens"], pb["tokens"]
    if min(len(ta), len(tb)) < MIN_CONTENT_TOKENS:
        return False
    inter = len(ta & tb)
    return (inter / len(ta | tb) >= JACCARD_THRESHOLD
            or inter / min(len(ta), len(tb)) >= CONTAINMENT_THRESHOLD)


def _pool_sources(members: List[Dict]) -> List[Dict]:
    seen, pooled = set(), []
    for m in members:
        for s in m.get("sources", []) or []:
            key = s.get("url") or s.get("title")
            if key and key not in seen:
                seen.add(key)
                pooled.append({**s, "id": len(pooled) + 1})
    return pooled


def merge_duplicate_claims(facts: List[Dict]) -> List[Dict]:
    """Greedy clustering. Singletons pass through untouched; merged facts carry
    `origins` (one entry per sub_question) and `merged_count` so verdicts can be
    expanded back afterwards."""
    clusters: List[List[Dict]] = []
    for fact in facts:
        for cluster in clusters:
            if claims_equivalent(cluster[0]["claim"], fact["claim"]):
                cluster.append(fact)
                break
        else:
            clusters.append([fact])

    merged = []
    for members in clusters:
        if len(members) == 1:
            merged.append(members[0])
            continue
        origins, seen_sq = [], set()
        for m in members:
            sq = m.get("sub_question")
            if sq not in seen_sq:
                seen_sq.add(sq)
                origins.append({"sub_question": sq, "claim": m["claim"]})
        best = max(members, key=lambda m: len(m["claim"]))
        merged.append({**members[0], "claim": best["claim"],
                       "sources": _pool_sources(members),
                       "origins": origins, "merged_count": len(members)})
    return merged


def expand_verdicts(results: List[Dict]) -> List[Dict]:
    """After fact-checking: give every originating sub-question its own copy of the
    shared verdict, so summarization still sees each planned topic's claims."""
    expanded = []
    for r in results:
        origins = r.get("origins")
        if not origins:
            expanded.append(r)
            continue
        base = {k: v for k, v in r.items() if k != "origins"}
        for o in origins:
            copy = {**base, "claim": o["claim"]}
            if o["sub_question"] is not None:
                copy["sub_question"] = o["sub_question"]
            expanded.append(copy)
    return expanded

Writing tools/claim_merge.py


In [2]:
%%writefile tools/diversity.py
"""Source diversity helpers (Day 24): prefer DIFFERENT domains over MORE results."""
from collections import Counter
from typing import Dict, List

from tools.evidence import source_domain, LOW_QUALITY_DOMAINS

MAX_SOURCES_PER_QUERY = 4
MIN_DIVERSE_DOMAINS = 3     


def _domain(r: Dict) -> str:
    return source_domain({"url": r.get("url", ""), "title": r.get("title", "")})


def select_diverse(results: List[Dict], max_total: int = MAX_SOURCES_PER_QUERY,
                   max_per_domain: int = 1) -> List[Dict]:
    """results: ranked best-first, each {"title","url","content"}. Keeps at most
    `max_per_domain` per registrable domain and skips low-quality domains. If that
    leaves nothing, falls back to the top results so the agent still has text."""
    kept, per = [], Counter()
    for r in results:
        d = _domain(r)
        if not d or d in LOW_QUALITY_DOMAINS or per[d] >= max_per_domain:
            continue
        per[d] += 1
        kept.append(r)
        if len(kept) >= max_total:
            break
    return kept or results[:max_total]


def domain_count(results: List[Dict]) -> int:
    return len({d for d in map(_domain, results) if d and d not in LOW_QUALITY_DOMAINS})


def add_new_domains(primary: List[Dict], extra: List[Dict],
                    max_total: int = MAX_SOURCES_PER_QUERY) -> List[Dict]:
    """Top up `primary` with results from `extra` ONLY if their domain is new."""
    out = list(primary)
    seen = {_domain(r) for r in out}
    for r in extra:
        d = _domain(r)
        if len(out) >= max_total:
            break
        if d and d not in seen and d not in LOW_QUALITY_DOMAINS:
            seen.add(d)
            out.append(r)
    return out

Writing tools/diversity.py


In [3]:
%%writefile tools/llm_clients.py
import os
import time
import threading
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient
from ddgs import DDGS

from tools.diversity import (select_diverse, add_new_domains, domain_count,
                             MAX_SOURCES_PER_QUERY, MIN_DIVERSE_DOMAINS)

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")
DEMO_MODE = os.getenv("DEMO_MODE", "false").lower() == "true"

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)

MAX_CONCURRENT_SEARCHES = 1 if DEMO_MODE else 2
_search_semaphore = threading.Semaphore(MAX_CONCURRENT_SEARCHES)

# Day 24: fetch a wider candidate pool, but KEEP only distinct domains.
CANDIDATE_POOL = 8


def is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            raise
    raise last_error


def _mark_snippets(snippets) -> str:
    """Tags each snippet [S1], [S2]... so extraction can say which source states a claim (Day 23)."""
    return " ".join(f"[S{i}] {text[:300]}" for i, text in enumerate(snippets, start=1))


def _package(results, provider: str) -> dict:
    """results: [{"title","url","content"}] -> the dict the agents expect."""
    sources = [{"id": i, "title": r.get("title", ""), "url": r.get("url", "")}
               for i, r in enumerate(results, start=1)]
    answer = _mark_snippets([r.get("content", "") for r in results])
    return {"answer": answer, "sources": sources, "provider": provider}


def _ddg_candidates(query: str, n: int = CANDIDATE_POOL) -> list:
    raw = list(DDGS().text(query, max_results=n))
    return [{"title": r.get("title", ""), "url": r.get("href") or r.get("url", ""),
             "content": r.get("body", "")} for r in raw]


def _search_duckduckgo(query: str) -> dict:
    picked = select_diverse(_ddg_candidates(query), max_total=MAX_SOURCES_PER_QUERY)
    return _package(picked, "duckduckgo")


def research_with_fallback(query: str) -> dict:
    """Gemini -> Tavily -> DuckDuckGo, concurrency-limited by a semaphore."""
    with _search_semaphore:
        try:
            resp = gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=query,
                config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
            )
            sources = []
            if resp.candidates[0].grounding_metadata:
                sources = [
                    {"id": i, "title": c.web.title, "url": c.web.uri}
                    for i, c in enumerate(resp.candidates[0].grounding_metadata.grounding_chunks, start=1)
                ]
            # Gemini's answer is one synthesized text, so it has no per-snippet tags.
            return {"answer": resp.text, "sources": sources, "provider": "gemini"}
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            print(f" Gemini unavailable ({type(e).__name__}), falling back to Tavily...")

        try:
            raw = tavily_client.search(query=query, max_results=CANDIDATE_POOL)["results"]
            candidates = [{"title": r.get("title", ""), "url": r.get("url", ""),
                           "content": r.get("content", "")} for r in raw]
            picked = select_diverse(candidates, max_total=MAX_SOURCES_PER_QUERY)
            provider = "tavily"
            # Too few distinct domains? Top up from DuckDuckGo, but only with domains we don't have.
            if domain_count(picked) < MIN_DIVERSE_DOMAINS:
                try:
                    topped = add_new_domains(picked, _ddg_candidates(query), MAX_SOURCES_PER_QUERY)
                    if len(topped) > len(picked):
                        picked, provider = topped, "tavily+duckduckgo"
                except Exception as e:
                    print(f"DuckDuckGo top-up skipped ({type(e).__name__})")
            return _package(picked, provider)
        except Exception as e:
            print(f" Tavily unavailable ({type(e).__name__}), falling back to DuckDuckGo...")

        try:
            return _search_duckduckgo(query)
        except Exception as e:
            print(f" DuckDuckGo also unavailable ({type(e).__name__})")
            return {"answer": "", "sources": [], "provider": "none"}

Overwriting tools/llm_clients.py


In [4]:
%%writefile agents/fact_checking_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event
from tools.evidence import independent_source_count, MIN_INDEPENDENT_SOURCES
from tools.claim_merge import merge_duplicate_claims, expand_verdicts

AGENT_NAME = "fact_checking_agent"
MAX_FACTS_PER_BATCH = 15


def _build_fact_check_prompt(facts: List[Dict]) -> str:
    numbered = "\n".join(
        f'{i}. "{f["claim"]}" (independent sources: {independent_source_count(f.get("sources", []))})'
        for i, f in enumerate(facts)
    )
    return f"""You are fact-checking a set of claims gathered during research. For EACH
numbered claim below, classify it as exactly one of:

- CONFIRMED: at least {MIN_INDEPENDENT_SOURCES} independent sources (see the count) and nothing in this list contradicts it
- DISPUTED: contradicts another claim in this list
- UNVERIFIED: fewer than {MIN_INDEPENDENT_SOURCES} independent sources, and no contradiction

Claims:
{numbered}

Respond with ONLY a JSON array: [{{"index": 0, "status": "CONFIRMED", "reasoning": "one short sentence"}}, ...]
"""


def _parse_fact_check_response(raw_text: str, expected_count: int) -> List[Dict]:
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if not isinstance(parsed, list):
            return []
        results = {}
        for item in parsed:
            idx = item.get("index")
            if isinstance(idx, int) and 0 <= idx < expected_count:
                status = item.get("status", "UNVERIFIED").upper()
                if status not in ("CONFIRMED", "DISPUTED", "UNVERIFIED"):
                    status = "UNVERIFIED"
                results[idx] = {"status": status, "reasoning": item.get("reasoning", "")}
        return [results.get(i, {"status": "UNVERIFIED", "reasoning": "not returned by model"})
                for i in range(expected_count)]
    except json.JSONDecodeError:
        return []


def apply_evidence_gate(fact: Dict, verdict: Dict) -> Dict:
    """Deterministic check applied AFTER the LLM verdict: a claim cannot stay CONFIRMED
    unless it has enough independent source domains, whatever the model said."""
    count = independent_source_count(fact.get("sources", []))
    status, reasoning = verdict["status"], verdict["reasoning"]
    downgraded = False
    if status == "CONFIRMED" and count < MIN_INDEPENDENT_SOURCES:
        status = "UNVERIFIED"
        reasoning = f"[downgraded: {count} independent source(s), need {MIN_INDEPENDENT_SOURCES}] {reasoning}"
        downgraded = True
    return {**fact, "status": status, "reasoning": reasoning,
            "independent_sources": count, "downgraded": downgraded}


def fact_check_batch(facts: List[Dict]) -> List[Dict]:
    if not facts:
        return []
    try:
        raw_output = call_llm(_build_fact_check_prompt(facts))
        verdicts = _parse_fact_check_response(raw_output, len(facts))
        if not verdicts:
            raise ValueError("fact-check response could not be parsed")
    except Exception as e:
        log_agent_event(AGENT_NAME, "batch_failed", error=f"{type(e).__name__}: {e}", batch_size=len(facts))
        verdicts = [{"status": "UNVERIFIED", "reasoning": "fact-check call failed; defaulted to unverified"}
                    for _ in facts]

    checked = [apply_evidence_gate(f, v) for f, v in zip(facts, verdicts)]

    counts: Dict[str, int] = {}
    for c in checked:
        counts[c["status"]] = counts.get(c["status"], 0) + 1
    log_agent_event(AGENT_NAME, "batch_done", downgraded=sum(1 for c in checked if c["downgraded"]), **counts)
    return checked


def fact_check_facts(facts: List[Dict]) -> List[Dict]:
    all_checked = []
    for i in range(0, len(facts), MAX_FACTS_PER_BATCH):
        all_checked.extend(fact_check_batch(facts[i:i + MAX_FACTS_PER_BATCH]))
    return all_checked


def fact_checking_node(state: Dict) -> Dict:
    extracted_facts = state.get("extracted_facts", [])
    if not extracted_facts:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no extracted_facts to check"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    # Day 24: check each distinct claim once, with its sources pooled across sub-questions,
    # then hand every originating sub-question its own copy of the verdict.
    merged = merge_duplicate_claims(extracted_facts)
    log_agent_event(AGENT_NAME, "claims_merged", facts_in=len(extracted_facts), distinct_claims=len(merged))

    checked = expand_verdicts(fact_check_facts(merged))
    return {
        "fact_check_results": checked,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "facts_checked": len(merged),
                             "facts_in": len(extracted_facts)}],
    }

Overwriting agents/fact_checking_agent.py


In [5]:
%%writefile tools/metrics.py
"""Run-level observability: tokens, latency and errors per node, plus evidence stats."""
import json
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler

RATE_LIMIT_MARKERS = ("429", "rate limit", "rate_limit")


def peak_tokens_in_window(events, window=60.0):
    """events: [(timestamp, output_tokens)]. Max token sum inside any `window`-second span."""
    events = sorted(events)
    best = cur = lo = 0
    for hi in range(len(events)):
        cur += events[hi][1]
        while events[hi][0] - events[lo][0] > window:
            cur -= events[lo][1]
            lo += 1
        best = max(best, cur)
    return best


class RunMetrics(BaseCallbackHandler):
    """Pass as config={"callbacks": [metrics]} to graph.invoke."""

    def __init__(self):
        self._lock = threading.Lock()
        self._llm_runs = {}    # run_id -> (node, start)
        self._node_runs = {}   # run_id -> (node, start)
        self.llm_calls = []
        self.node_times = defaultdict(list)
        self.node_errors = Counter()
        self.output_events = []   # (timestamp, output_tokens)
        self.started = time.time()

    @staticmethod
    def _node(metadata):
        return (metadata or {}).get("langgraph_node", "unknown")

    # ---- LLM calls ----
    def on_chat_model_start(self, serialized, messages, *, run_id, metadata=None, **kwargs):
        with self._lock:
            self._llm_runs[run_id] = (self._node(metadata), time.time())

    def on_llm_start(self, serialized, prompts, *, run_id, metadata=None, **kwargs):
        with self._lock:
            self._llm_runs[run_id] = (self._node(metadata), time.time())

    def on_llm_end(self, response, *, run_id, **kwargs):
        now = time.time()
        with self._lock:
            node, start = self._llm_runs.pop(run_id, ("unknown", now))
        inp = out = reasoning = 0
        try:
            msg = response.generations[0][0].message
            um = getattr(msg, "usage_metadata", None) or {}
            inp = um.get("input_tokens", 0)
            out = um.get("output_tokens", 0)
            reasoning = (um.get("output_token_details") or {}).get("reasoning", 0)
        except (IndexError, AttributeError):
            pass
        if not out:
            tu = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
            inp = inp or tu.get("prompt_tokens", 0)
            out = tu.get("completion_tokens", 0)
        with self._lock:
            self.llm_calls.append({"node": node, "input": inp, "output": out,
                                   "reasoning": reasoning, "seconds": now - start,
                                   "error": None})
            self.output_events.append((now, out))

    def on_llm_error(self, error, *, run_id, **kwargs):
        now = time.time()
        with self._lock:
            node, start = self._llm_runs.pop(run_id, ("unknown", now))
        msg = str(error).lower()
        kind = "rate_limit" if any(m in msg for m in RATE_LIMIT_MARKERS) else "other"
        with self._lock:
            self.llm_calls.append({"node": node, "input": 0, "output": 0,
                                   "reasoning": 0, "seconds": now - start,
                                   "error": kind})

    # ---- node latency (a node is a chain whose name equals its langgraph_node) ----
    def on_chain_start(self, serialized, inputs, *, run_id, metadata=None, **kwargs):
        node = (metadata or {}).get("langgraph_node")
        if node and kwargs.get("name") == node:
            with self._lock:
                self._node_runs[run_id] = (node, time.time())

    def _finish_node(self, run_id, failed=False):
        with self._lock:
            entry = self._node_runs.pop(run_id, None)
            if entry:
                node, start = entry
                self.node_times[node].append(time.time() - start)
                if failed:
                    self.node_errors[node] += 1

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        self._finish_node(run_id)

    def on_chain_error(self, error, *, run_id, **kwargs):
        self._finish_node(run_id, failed=True)

    # ---- summary ----
    def summary(self):
        nodes = defaultdict(lambda: {"calls": 0, "input": 0, "output": 0, "reasoning": 0,
                                     "rate_limited": 0, "other_errors": 0,
                                     "llm_seconds": 0.0, "max_call_output": 0})
        for c in self.llm_calls:
            n = nodes[c["node"]]
            n["calls"] += 1
            n["input"] += c["input"]
            n["output"] += c["output"]
            n["reasoning"] += c["reasoning"]
            n["llm_seconds"] += c["seconds"]
            n["max_call_output"] = max(n["max_call_output"], c["output"])
            if c["error"] == "rate_limit":
                n["rate_limited"] += 1
            elif c["error"]:
                n["other_errors"] += 1
        for node, times in self.node_times.items():
            nodes[node]["node_runs"] = len(times)
            nodes[node]["node_seconds"] = sum(times)
            nodes[node]["slowest_run"] = max(times)
            nodes[node]["node_errors"] = self.node_errors[node]
        return {
            "total_seconds": time.time() - self.started,
            "total_output_tokens": sum(c["output"] for c in self.llm_calls),
            "total_reasoning_tokens": sum(c["reasoning"] for c in self.llm_calls),
            "peak_output_tokens_60s": peak_tokens_in_window(self.output_events),
            "rate_limit_errors": sum(1 for c in self.llm_calls if c["error"] == "rate_limit"),
            "nodes": {k: dict(v) for k, v in nodes.items()},
        }


def evidence_stats(state):
    """Evidence picture from the final state: claims before/after merging, cited vs
    independent sources per claim, and search-provider mix."""
    cited, independent, status = Counter(), Counter(), Counter()
    results = state.get("fact_check_results", []) or []
    for f in results:
        cited[len(f.get("sources") or f.get("source_ids") or [])] += 1
        independent[f.get("independent_sources", "n/a")] += 1
        status[f.get("status") or f.get("verdict") or "n/a"] += 1
    extracted = state.get("extracted_facts", []) or []
    merged_groups = [f for f in results if f.get("merged_count", 1) > 1]
    providers = Counter(r.get("provider", "n/a") for r in (state.get("search_results", []) or [])
                        if isinstance(r, dict))
    return {"extracted_claims": len(extracted),
            "claims_after_merge_expand": len(results),
            "claims_that_were_merged": len(merged_groups),
            "cited_per_claim": dict(sorted(cited.items())),
            "independent_per_claim": dict(independent),
            "status": dict(status),
            "search_providers": dict(providers)}


def save_run(metrics, state, label="run"):
    Path("data/runs").mkdir(parents=True, exist_ok=True)
    path = Path("data/runs") / f"{datetime.now():%Y%m%d_%H%M%S}_{label}.json"
    path.write_text(json.dumps({"metrics": metrics.summary(),
                                "evidence": evidence_stats(state)}, indent=2, default=str))
    return path


def print_run_report(metrics, state):
    s = metrics.summary()
    print(f"Total: {s['total_seconds']:.0f}s | output tokens: {s['total_output_tokens']} "
          f"(reasoning: {s['total_reasoning_tokens']}) | 429s: {s['rate_limit_errors']}")
    print(f"Peak output tokens in any 60s window: {s['peak_output_tokens_60s']} (Groq limit: 1000)\n")
    print(f"{'node':<22}{'calls':>6}{'out':>7}{'reason':>8}{'max/call':>9}{'429':>5}{'node_s':>8}")
    for name, n in s["nodes"].items():
        print(f"{name:<22}{n['calls']:>6}{n['output']:>7}{n['reasoning']:>8}"
              f"{n['max_call_output']:>9}{n['rate_limited']:>5}{n.get('node_seconds', 0):>8.1f}")
    ev = evidence_stats(state)
    print("\nEvidence")
    print(f"  claims extracted: {ev['extracted_claims']} -> after merge/expand: "
          f"{ev['claims_after_merge_expand']} ({ev['claims_that_were_merged']} rows from merged groups)")
    print("  cited sources per claim:      ", ev["cited_per_claim"])
    print("  independent domains per claim:", ev["independent_per_claim"])
    print("  status:                       ", ev["status"])
    print("  search providers:             ", ev["search_providers"])

Writing tools/metrics.py


In [6]:
%%writefile tests/test_claim_merge.py
from tools.claim_merge import claims_equivalent, merge_duplicate_claims, expand_verdicts


def src(url, i=1):
    return {"id": i, "title": url, "url": url}


A = "Redwood Materials recovers 95% of cobalt and nickel from used lithium-ion batteries"
A2 = "Redwood Materials recovers 95% of the cobalt and nickel in used lithium-ion batteries"


def test_rephrased_claims_are_equivalent():
    assert claims_equivalent(A, A2)


def test_different_numbers_never_merge():
    assert not claims_equivalent(A, A.replace("95%", "70%"))


def test_negation_never_merges():
    assert not claims_equivalent("Solid-state cells do not catch fire during nail penetration tests easily",
                                 "Solid-state cells catch fire during nail penetration tests easily")


def test_short_claims_never_merge():
    assert not claims_equivalent("Claim A", "Claim A")


def test_merge_pools_sources_and_tracks_origins():
    facts = [
        {"claim": A, "sub_question": "Q1", "sources": [src("https://redwoodmaterials.com/x")]},
        {"claim": A2, "sub_question": "Q2", "sources": [src("https://reuters.com/y")]},
        {"claim": "Totally different statement about CATL sodium cells and their price range here",
         "sub_question": "Q1", "sources": [src("https://a.com/1")]},
    ]
    merged = merge_duplicate_claims(facts)
    assert len(merged) == 2
    m = merged[0]
    assert m["merged_count"] == 2
    assert {s["url"] for s in m["sources"]} == {"https://redwoodmaterials.com/x", "https://reuters.com/y"}
    assert [s["id"] for s in m["sources"]] == [1, 2]
    assert {o["sub_question"] for o in m["origins"]} == {"Q1", "Q2"}
    assert "origins" not in merged[1]            # singleton untouched


def test_same_source_url_not_double_counted():
    facts = [{"claim": A, "sub_question": "Q1", "sources": [src("https://a.com/p")]},
             {"claim": A2, "sub_question": "Q2", "sources": [src("https://a.com/p")]}]
    assert len(merge_duplicate_claims(facts)[0]["sources"]) == 1


def test_duplicates_inside_one_subquestion_collapse():
    facts = [{"claim": A, "sub_question": "Q1", "sources": [src("https://a.com/p")]},
             {"claim": A2, "sub_question": "Q1", "sources": [src("https://b.com/p")]}]
    merged = merge_duplicate_claims(facts)
    assert len(merged[0]["origins"]) == 1
    out = expand_verdicts([{**merged[0], "status": "CONFIRMED"}])
    assert len(out) == 1


def test_expand_gives_every_subquestion_the_verdict():
    facts = [{"claim": A, "sub_question": "Q1", "sources": [src("https://a.com/p")]},
             {"claim": A2, "sub_question": "Q2", "sources": [src("https://b.com/p")]}]
    merged = merge_duplicate_claims(facts)
    out = expand_verdicts([{**merged[0], "status": "CONFIRMED"}])
    assert sorted(r["sub_question"] for r in out) == ["Q1", "Q2"]
    assert all(r["status"] == "CONFIRMED" and "origins" not in r for r in out)

Writing tests/test_claim_merge.py


In [7]:
%%writefile tests/test_fact_check_merge.py
import json
import agents.fact_checking_agent as fc

A = "Redwood Materials recovers 95% of cobalt and nickel from used lithium-ion batteries"
A2 = "Redwood Materials recovers 95% of the cobalt and nickel in used lithium-ion batteries"


def _facts():
    return [
        {"claim": A, "sub_question": "Q1", "sources": [{"id": 1, "title": "x", "url": "https://a.com/1"}]},
        {"claim": A2, "sub_question": "Q2", "sources": [{"id": 1, "title": "y", "url": "https://b.org/1"}]},
    ]


def test_merged_claim_is_checked_once_and_confirmed_with_pooled_sources(monkeypatch):
    prompts = []

    def fake(prompt):
        prompts.append(prompt)
        return json.dumps([{"index": 0, "status": "CONFIRMED", "reasoning": "two domains"}])

    monkeypatch.setattr(fc, "call_llm", fake)
    out = fc.fact_checking_node({"extracted_facts": _facts()})["fact_check_results"]
    assert len(prompts) == 1 and "independent sources: 2" in prompts[0]
    assert sorted(r["sub_question"] for r in out) == ["Q1", "Q2"]
    assert all(r["status"] == "CONFIRMED" and r["independent_sources"] == 2 for r in out)


def test_gate_still_downgrades_merged_claim_from_one_domain(monkeypatch):
    facts = _facts()
    facts[1]["sources"][0]["url"] = "https://a.com/2"      # same domain as the first
    monkeypatch.setattr(fc, "call_llm", lambda p: json.dumps(
        [{"index": 0, "status": "CONFIRMED", "reasoning": "x"}]))
    out = fc.fact_checking_node({"extracted_facts": facts})["fact_check_results"]
    assert all(r["status"] == "UNVERIFIED" and r["downgraded"] for r in out)

Writing tests/test_fact_check_merge.py


In [8]:
%%writefile tests/test_metrics.py
import uuid
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, LLMResult
from tools.metrics import RunMetrics, peak_tokens_in_window, evidence_stats


def _resp(out, inp=10, reasoning=0):
    msg = AIMessage(content="x", usage_metadata={
        "input_tokens": inp, "output_tokens": out, "total_tokens": inp + out,
        "output_token_details": {"reasoning": reasoning}})
    return LLMResult(generations=[[ChatGeneration(message=msg)]])


def test_peak_window_math():
    assert peak_tokens_in_window([(0, 400), (30, 400), (59, 100), (200, 900)], 60) == 900
    assert peak_tokens_in_window([(0, 400), (30, 400), (59, 100), (100, 900)], 60) == 1000  # 59s + 100s fit in one window
    assert peak_tokens_in_window([(0, 500), (61, 500)], 60) == 500
    assert peak_tokens_in_window([], 60) == 0


def test_llm_call_attributed_to_node_with_reasoning():
    m, rid = RunMetrics(), uuid.uuid4()
    m.on_chat_model_start({}, [], run_id=rid, metadata={"langgraph_node": "summarization"})
    m.on_llm_end(_resp(300, reasoning=120), run_id=rid)
    n = m.summary()["nodes"]["summarization"]
    assert n["output"] == 300 and n["reasoning"] == 120 and n["calls"] == 1


def test_parallel_calls_keep_separate_attribution():
    m = RunMetrics()
    a, b = uuid.uuid4(), uuid.uuid4()
    m.on_chat_model_start({}, [], run_id=a, metadata={"langgraph_node": "web_search_single"})
    m.on_chat_model_start({}, [], run_id=b, metadata={"langgraph_node": "web_search_single"})
    m.on_llm_end(_resp(50), run_id=a)
    m.on_llm_end(_resp(70), run_id=b)
    assert m.summary()["nodes"]["web_search_single"]["output"] == 120


def test_rate_limit_error_counted():
    m, rid = RunMetrics(), uuid.uuid4()
    m.on_chat_model_start({}, [], run_id=rid, metadata={"langgraph_node": "fact_checking"})
    m.on_llm_error(Exception("Error code: 429 - rate_limit_exceeded"), run_id=rid)
    assert m.summary()["rate_limit_errors"] == 1


def test_node_latency_only_counts_the_node_chain():
    m, rid, inner = RunMetrics(), uuid.uuid4(), uuid.uuid4()
    md = {"langgraph_node": "supervisor"}
    m.on_chain_start({}, {}, run_id=rid, metadata=md, name="supervisor")
    m.on_chain_start({}, {}, run_id=inner, metadata=md, name="RunnableSequence")
    m.on_chain_end({}, run_id=inner)
    m.on_chain_end({}, run_id=rid)
    assert m.summary()["nodes"]["supervisor"]["node_runs"] == 1


def test_evidence_stats_distribution_and_merge_count():
    state = {"extracted_facts": [1, 2, 3, 4],
             "fact_check_results": [
                 {"sources": [1, 2], "independent_sources": 1, "status": "UNVERIFIED"},
                 {"sources": [1], "independent_sources": 1, "status": "UNVERIFIED"},
                 {"sources": [1, 2, 3], "independent_sources": 2, "status": "CONFIRMED", "merged_count": 2}]}
    ev = evidence_stats(state)
    assert ev["cited_per_claim"] == {1: 1, 2: 1, 3: 1}
    assert ev["independent_per_claim"] == {1: 2, 2: 1}
    assert ev["claims_that_were_merged"] == 1 and ev["extracted_claims"] == 4

Writing tests/test_metrics.py


In [10]:
from graph.build_graph import build_graph
from tools.metrics import RunMetrics, save_run, print_run_report

app = build_graph()
metrics = RunMetrics()

query = "Compare battery recycling approaches of Tesla, Redwood Materials and CATL"  # or any fresh query

initial_state = {
    "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}

result = app.invoke(initial_state, config={"callbacks": [metrics]})
print_run_report(metrics, result)
print("saved:", save_run(metrics, result, label="day24"))

2026-11-01 13:16:02,254 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...


2026-11-01 13:16:07,377 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-01 13:16:07,750 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0


 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...


2026-11-01 13:16:09,731 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-01 13:16:10,371 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-01 13:16:10,382 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-11-01 13:16:12,468 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the primary technical processes and recovery rates for lithium, cobalt, and nickel in the recycling facilities operated by Tesla, Redwood Materials, and CATL? facts_found=18
2026-11-01 13:16:13,054 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do the business models and supply chain integration strategies of Tesla, Redwood Materials, and CATL differ in sourcing end-of-life batteries for recycling? facts_found=5
2026-11-01 13:17:05,099 | INFO | research_system | [data_extrac

Total: 210s | output tokens: 4397 (reasoning: 0) | 429s: 0
Peak output tokens in any 60s window: 1692 (Groq limit: 1000)

node                   calls    out  reason max/call  429  node_s
supervisor                 1    127       0      127    0     3.5
data_extraction            4   1692       0      841    0    55.3
fact_checking              3   1129       0      489    0    65.9
summarization              4    626       0      186    0    54.3
report_generator           1    823       0      823    0    22.9
web_search_single          0      0       0        0    0    25.9

Evidence
  claims extracted: 36 -> after merge/expand: 34 (2 rows from merged groups)
  cited sources per claim:       {1: 34}
  independent domains per claim: {1: 34}
  status:                        {'UNVERIFIED': 34}
  search providers:              {'tavily': 4}
saved: data\runs\20261101_131928_day24.json


In [11]:
!python -m pytest tests/ -q
with open(".gitignore", "a") as f:
    f.write("\ndata/runs/\ndata/checkpoints.db-shm\ndata/checkpoints.db-wal\n")

........................................................................ [ 81%]
................                                                         [100%]
88 passed in 3.81s


In [12]:
import re
from tools.claim_merge import _profile
from tools.evidence import independent_source_count

sr = result["search_results"]
print("search_results keys:", list(sr[0].keys()) if sr else None)

pool = []   # every snippet from every sub-question
for r in sr:
    parts = re.split(r"\[S(\d+)\]", r.get("answer") or "")
    texts = {int(parts[i]): parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)}
    for s in r.get("sources", []):
        if s["id"] in texts:
            pool.append({"url": s["url"], "title": s["title"], "text": texts[s["id"]]})
print("snippets in pool:", len(pool))

for f in result["fact_check_results"][:8]:
    p = _profile(f["claim"])
    support = []
    for sn in pool:
        sp = _profile(sn["text"])
        overlap = len(p["tokens"] & sp["tokens"]) / max(1, len(p["tokens"]))
        if p["numbers"] <= sp["numbers"] and overlap >= 0.5:
            support.append(sn)
    print("\nCLAIM:", f["claim"][:110])
    print("  cited:", [s["url"] for s in f.get("sources", [])])
    print(f"  supported by {len(support)} snippet(s), {independent_source_count(support)} independent domain(s):")
    for sn in support:
        print("   -", sn["url"])

search_results keys: ['sub_question', 'answer', 'sources', 'provider', 'status', 'latency_s']
snippets in pool: 16

CLAIM: Hydrometallurgical processing achieves a 95% recovery rate for lithium.
  cited: ['https://invrecovery.org/ev-battery-recycling-investment-recovery-2026-playbook']
  supported by 1 snippet(s), 1 independent domain(s):
   - https://invrecovery.org/ev-battery-recycling-investment-recovery-2026-playbook

CLAIM: Hydrometallurgical processing achieves a 97% recovery rate for nickel.
  cited: ['https://invrecovery.org/ev-battery-recycling-investment-recovery-2026-playbook']
  supported by 1 snippet(s), 1 independent domain(s):
   - https://invrecovery.org/ev-battery-recycling-investment-recovery-2026-playbook

CLAIM: Hydrometallurgy holds roughly 70% of global recycling capacity.
  cited: ['https://invrecovery.org/ev-battery-recycling-investment-recovery-2026-playbook']
  supported by 1 snippet(s), 1 independent domain(s):
   - https://invrecovery.org/ev-battery-recyclin

In [ ]:
from collections import Counter

def supporters(claim):
    p = _profile(claim)
    out = []
    for sn in pool:
        sp = _profile(sn["text"])
        if p["numbers"] <= sp["numbers"] and len(p["tokens"] & sp["tokens"]) / max(1, len(p["tokens"])) >= 0.5:
            out.append(sn)
    return out

dist = Counter(independent_source_count(supporters(f["claim"])) for f in result["fact_check_results"])
print("claims by number of independent supporting domains:", dict(sorted(dist.items())))